# SNAKE WITH PORTALS!

needed libraries

In [ ]:
import os
import sys
import asyncio
import websockets
import json
import webbrowser
import time
from IPython.display import display, HTML
from functools import partial
from copy import deepcopy
import heapq
import itertools
from queue import deque
from game import *

Configs

In [ ]:
map_dir_path = "maps"

DIRECTIONS = {
'U': (-1, 0),
'D': (1, 0),
'L': (0, -1),
'R': (0, 1)
}

DFS_limit = None
IDS_max_depth = 1000
time_limit = 100.0


let's see how to execute one move

In [ ]:
def render_notebook(game):

    for r in range(game.board.height):
        row = []

        for c in range(game.board.width):
            pos = (r,c)
            if pos == game.snake.body[0]:
                row.append("S")
            elif pos in game.snake.body[1:]:
                row.append("B")
            elif pos in game.board.walls:
                row.append("W")
            elif pos in game.board.apples:
                row.append("A")
            elif pos in game.board.p_portals:
                row.append("P")
            elif pos in game.board.q_portals:
                row.append("Q")
            else:
                row.append(".")

        print(" ".join(row))

    print()


game = Game(map_dir_path + '/' + "map01.txt")
game.move("R")
render_notebook(game)

W W W W W W W W W W
W . S . . W . . A W
W . W W . W . W . W
W . W . . . . W . W
W . W . W W . W . W
W . . . W . . . . W
W W W . W W W W . W
W . . . . . . W . W
W A W W W W . . . W
W W W W W W W W W W



now one step further!

let's see how to execute a path

In [ ]:
def show_path(game, path, delay=0.5):

    print("Initial state:")
    render_notebook(game)

    for step, move in enumerate(path, 1):

        game.move(move)

        print(f"Step {step}  Move: {move}")
        render_notebook(game)

        time.sleep(delay)

        if game.state != "PLAYING":
            print("Game ended:", game.state)
            break


game = Game(map_dir_path + '/' + "map01.txt")

path = ["U","U","R","R","D","D","D","L"]

show_path(game, path)

Initial state:
W W W W W W W W W W
W S . . . W . . A W
W . W W . W . W . W
W . W . . . . W . W
W . W . W W . W . W
W . . . W . . . . W
W W W . W W W W . W
W . . . . . . W . W
W A W W W W . . . W
W W W W W W W W W W

Step 1  Move: U
W S W W W W W W W W
W . . . . W . . A W
W . W W . W . W . W
W . W . . . . W . W
W . W . W W . W . W
W . . . W . . . . W
W W W . W W W W . W
W . . . . . . W . W
W A W W W W . . . W
W W W W W W W W W W

Game ended: LOSS


## Snake Needs Your HELP!!

---

Your time to code !

**step 1.** define your state,

 when traversing the game graph, what would be apporpiate for nodes to contain?




In [ ]:
# ------ TODO ------ 

state = {}

**step 2.** now you need to calculate the next state,

 think in the game graph when snake makes a move, what would the next
child node be?

In [ ]:
# ------ TODO ------ 

def next_state(state, board, action):
    """return next state according to the board (look at Map class) and current state and the chosen action"""
    pass

**step 3.** implement solvers!

if the solver finds a path then return path and number of visited nodes

if the solver doesn't find a path or exceeds the time limit return an empty list and number of visited nodes 

you can check if it exceeds the time limit using time.perf_counter() which returns systems current time and time_limit variable defined in config above

BFS

In [ ]:
def solve_bfs(game):
    pass   



DFS

In [ ]:
def solve_dfs(game):
    pass

IDS

ids must iteratively search on depths up to max_depth

In [ ]:
def solve_ids(game, max_depth=IDS_max_depth):
    pass

Heuristics

in this cell, define your heuristic functions.

**DO NOT FORGET TO ADD YOUR HEURISTIC FUNCTIONS' NAME TO THE heuristics LIST BELOW, OTHERWISE THE GUI WOULDN'T RECOGNIZE IT.**

In [ ]:
# ------ TODO ------ 

def dummy_heuristic(state, board):
    return 0

heuristics = [dummy_heuristic] # add your heuristics here.
heuristic_map = {func.__name__: func for func in heuristics}

A* and weighted A*

In [ ]:
def solve_a_star(game, heuristic, weight = 1):
    pass

Time to see our snake ! 

In [ ]:
def is_running_in_jupyter():
    try:
        shell = get_ipython().__class__.__name__
        
        if shell == 'ZMQInteractiveShell':
            return True 
        elif shell == 'TerminalInteractiveShell':
            return False
        else:
            return False
            
    except NameError:
        return False

async def handler(websocket, game, shutdown_event):
    print("Browser connected.")

    await websocket.send(
            json.dumps(
                {
                    "type": "env",
                    "env": "jupyter" if is_running_in_jupyter() else "shell"
                }
            )
        )
    
    
    try:
        async for message in websocket:
            data = json.loads(message)
            msg_type = data.get("type")

            if msg_type == "init":
                game.reset(map_dir_path + '/' + data.get("map"))
                await websocket.send(json.dumps(game.get_state_dict()))

            elif msg_type == "move":
                key = data.get("key")
                game.move(key)
                await websocket.send(json.dumps(game.get_state_dict()))
                
            elif msg_type == "solve_ai":
                algo = data.get("algorithm")
                heuristic_name = heuristic_map.get(data.get("heuristic"))
                weight = None
                try:
                    weight = float(data.get("weight"))
                except:
                    pass

                game.reset(map_dir_path + '/' + data.get("map"))
                await websocket.send(json.dumps(game.get_state_dict()))
                
                moves = []
                start_time = time.perf_counter()
                
                if algo == "BFS":
                    moves, visited_nodes = solve_bfs(game)
                elif algo == "DFS":
                    moves, visited_nodes = solve_dfs(game, DFS_limit)
                elif algo == "IDS":
                    moves, visited_nodes = solve_ids(game, IDS_max_depth)
                elif algo == "A_STAR":
                    moves, visited_nodes = solve_a_star(game, heuristic_name, 1)
                elif algo == "WEIGHTED_A_STAR":
                    moves, visited_nodes = solve_a_star(game, heuristic_name, weight)
                    
                
                end_time = time.perf_counter()

                execution_time_ms = (end_time - start_time) * 1000
                
                await websocket.send(json.dumps({
                    "type": "ai_solution",
                    "moves": moves,
                    "executionTimeMs": execution_time_ms,
                    "visitedNodes": visited_nodes
                }))

            elif msg_type == "get-maps":
                map_files = [f for f in os.listdir(map_dir_path) if f.endswith('.txt')]

                await websocket.send(json.dumps({
                    "type": "maps",
                    "maps": map_files
                }))
                
            elif msg_type == "get-heuristics":
                await websocket.send(json.dumps({
                    "type": "heuristics",
                    "heuristics": list(heuristic_map.keys())
                }))

            elif msg_type == "print-logs":
                log_text = data.get("text")
                display(HTML(log_text))


    except websockets.exceptions.ConnectionClosed:
        print("Browser closed.")
        shutdown_event.set() 
    finally:
        print("Browser closed.")
        shutdown_event.set()

async def run_ui(game):

    file_path = os.path.abspath("index.html")
    if not os.path.isfile(file_path):
        print("index.html not found!!, make sure it's in the same path as this file")


    shutdown_event = asyncio.Event()
    bound_handler = partial(handler, game=game, shutdown_event=shutdown_event)

    print("Runnig server on port 8765...")
    async with websockets.serve(bound_handler, "localhost", 8765):
        webbrowser.open(f"file://{file_path}")
        await shutdown_event.wait()

In [ ]:
map_file_path = "maps/map01.txt" 
    
print("Welcome to Snake Portals!")
game_instance = Game(map_file_path)
await run_ui(game_instance)

Welcome to Snake Portals!
Runnig server on port 8765...
Browser connected.
Browser closed.
